<a href="https://colab.research.google.com/github/mlbyhoang/marketing-attribution/blob/main/notebooks/09_gru_training_model2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 09 – GRU with Attention – Model Training

This notebook trains and evaluates the GRU with Attention using the selected hyperparameters.

In [ ]:
# Load Libraries
import os
import json
import random
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from google.colab import userdata

from imblearn.over_sampling import RandomOverSampler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    log_loss
)

from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    Embedding,
    GRU,
    Dense,
    Lambda,
    Concatenate,
    Multiply
)
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

In [ ]:
# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Connect to Google Drive
drive.mount("/content/drive")

# Paths
filtered_path = userdata.get("MTA_FILTERED_PATH")
save_path = userdata.get("MTA_RESULTS_PATH")

os.makedirs(save_path, exist_ok=True)

Mounted at /content/drive


In [ ]:
# Load filtered journey data
df_filtered = pd.read_parquet(filtered_path)

print(f"Number of journeys: {len(df_filtered):,}")
print(f"Conversion rate: {df_filtered['order'].mean():.2%}")
print(f"Maximum journey length: {df_filtered['touchpoint_sum'].max()}")

,conversion_id,activity_time,activity,impression_count,click_count,interaction_time,interaction_number,campaign,interaction_type,placement,floodlight_variable_dimension21,activity_count
0,3800910923227433881,2026-02-11 17:55:06,ProductView_Server,0,3,2026-02-11 17:43:10,3,DART Search,Click,deinhandy.de GoogleAds Search,GA1.1.1833507673.1768199735,0
1,5459534759657986833,2026-02-20 10:11:46,Pageview_Server,0,3,2026-02-20 10:11:42,1,DART Search,Click,deinhandy.de GoogleAds Search,GA1.1.1682335648.1771578682,0
2,7173993360599741160,2026-02-10 21:05:52,Pageview_Server,0,3,2026-02-10 21:05:28,1,DART Search,Click,deinhandy.de GoogleAds Search,GA1.1.870244807.1769543999,0
3,7658577527247096958,2026-02-17 20:38:49,Pageview_Server,0,3,2026-02-17 14:22:36,2,DART Search,Click,deinhandy.de GoogleAds Search,GA1.1.239276648.1771334558,0
4,7729130632504957835,2026-02-16 09:46:32,Pageview_Server,0,3,2026-02-16 09:11:59,2,DART Search,Click,deinhandy.de GoogleAds Search,GA1.1.825061615.1769627264,0


In [ ]:
# Display first few rows
df_filtered.head()

In [ ]:
# Encode touchpoint sequences
all_touchpoints = [
    tp
    for seq in df_filtered["sequence"]
    for tp in seq
]

le = LabelEncoder()
le.fit(all_touchpoints)

print(f"Number of unique touchpoints: {len(le.classes_)}")
print(f"Touchpoint types: {le.classes_}")

# Reserve 0 for padding
X_seq_encoded = [
    le.transform(seq) + 1
    for seq in df_filtered["sequence"]
]

max_len = int(df_filtered["touchpoint_sum"].max())

X_seq_padded = np.array([
    np.pad(
        seq,
        (0, max_len - len(seq)),
        constant_values=0
    )
    for seq in X_seq_encoded
])

print(f"\nMaximum sequence length: {max_len}")
print(f"X_seq shape: {X_seq_padded.shape}")

# Sanity check
print(f"\nEncoded sequence:  {X_seq_padded[0]}")
print(f"Original sequence: {df_filtered['sequence'].iloc[0]}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11174533 entries, 0 to 11174532
Data columns (total 12 columns):
 #   Column                           Dtype 
---  ------                           ----- 
 0   conversion_id                    uint64
 1   activity_time                    object
 2   activity                         object
 3   impression_count                 int64 
 4   click_count                      int64 
 5   interaction_time                 object
 6   interaction_number               int64 
 7   campaign                         object
 8   interaction_type                 object
 9   placement                        object
 10  floodlight_variable_dimension21  object
 11  activity_count                   int64 
dtypes: int64(4), object(7), uint64(1)
memory usage: 1023.1+ MB


In [ ]:
# Define time features
time_feature_cols = [
    "interaction_hour_sin",
    "interaction_hour_cos",
    "interaction_weekday_sin",
    "interaction_weekday_cos",
    "tp_time_delta_min",
    "interaction_number",
    "is_click"
]

n_time_features = len(time_feature_cols)

# Build time feature sequences
def build_time_sequences(df, max_len, time_feature_cols):
    time_seqs = []

    for _, row in df.iterrows():
        time_vals = np.array([
            [d[col] for col in time_feature_cols]
            for d in row["seq_times"]
        ], dtype=np.float32)

        pad_len = max_len - len(time_vals)

        if pad_len > 0:
            time_vals = np.vstack([
                time_vals,
                np.zeros(
                    (pad_len, len(time_feature_cols)),
                    dtype=np.float32
                )
            ])
        else:
            time_vals = time_vals[:max_len]

        time_seqs.append(time_vals)

    return np.array(time_seqs, dtype=np.float32)


X_time = build_time_sequences(
    df_filtered,
    max_len,
    time_feature_cols
)

print(f"X_time shape: {X_time.shape}")

# Sanity check
print("\nFirst journey - first touchpoint:")
print(X_time[0, 0, :])

print("\nFirst journey - second time step:")
print(X_time[0, 1, :])

In [ ]:
# Numeric journey-level features and target
X_num = (
    df_filtered[["touchpoint_sum"]]
    .values
    .astype(np.float32)
)

y = (
    df_filtered["order"]
    .values
    .reshape(-1, 1)
    .astype(np.float32)
)

vocab_size = len(le.classes_) + 1
n_num_features = X_num.shape[1]

print(f"Vocabulary size:            {vocab_size}")
print(f"Number of time features:    {n_time_features}")
print(f"Number of numeric features: {n_num_features}")
print(f"Maximum sequence length:    {max_len}")

In [ ]:
# Train / validation / test split
indices = np.arange(len(df_filtered))

idx_trainval, idx_test = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y.ravel()
)

idx_train, idx_val = train_test_split(
    idx_trainval,
    test_size=0.125,
    random_state=42,
    stratify=y[idx_trainval].ravel()
)

# Sequence input
X_seq_train, X_seq_val, X_seq_test = (
    X_seq_padded[idx_train],
    X_seq_padded[idx_val],
    X_seq_padded[idx_test]
)

# Time-feature input
X_time_train, X_time_val, X_time_test = (
    X_time[idx_train],
    X_time[idx_val],
    X_time[idx_test]
)

# Numerical journey-level input
X_num_train, X_num_val, X_num_test = (
    X_num[idx_train],
    X_num[idx_val],
    X_num[idx_test]
)

# Target
y_train, y_val, y_test = (
    y[idx_train],
    y[idx_val],
    y[idx_test]
)

print(f"Train:      {len(y_train)}")
print(f"Validation: {len(y_val)}")
print(f"Test:       {len(y_test)}")

print(f"\nConversion rate - Train:      {y_train.mean():.2%}")
print(f"Conversion rate - Validation: {y_val.mean():.2%}")
print(f"Conversion rate - Test:       {y_test.mean():.2%}")

In [ ]:
# Scale numerical features using training data only
scaler_num = StandardScaler()

X_num_train = scaler_num.fit_transform(X_num_train)
X_num_val = scaler_num.transform(X_num_val)
X_num_test = scaler_num.transform(X_num_test)

In [ ]:
# Oversample training data while keeping all model inputs aligned
def oversample_sequences(
    X_seq,
    X_time,
    X_num,
    y,
    sampling_strategy
):
    indices = np.arange(len(y)).reshape(-1, 1)

    ros = RandomOverSampler(
        sampling_strategy=sampling_strategy,
        random_state=42
    )

    indices_resampled, y_resampled = ros.fit_resample(
        indices,
        y.ravel()
    )

    indices_resampled = indices_resampled.ravel()

    return (
        X_seq[indices_resampled],
        X_time[indices_resampled],
        X_num[indices_resampled],
        y_resampled.reshape(-1, 1).astype(np.float32)
    )

In [ ]:
# Load selected imbalance parameters
save_path = userdata.get("MTA_RESULTS_PATH")

with open(
    f"{save_path}best_imbalance_params.json",
    "r"
) as f:
    best_params = json.load(f)

selected_class_weight = best_params["class_weight_1"]
selected_sampling_strategy = best_params["sampling_strategy"]

best_cw_dict = (
    {0: 1.0, 1: selected_class_weight}
    if selected_class_weight is not None
    else None
)

print("===== Selected Imbalance Parameters =====")
print(f"Class weights: {best_cw_dict}")
print(f"Sampling strategy: {selected_sampling_strategy}")

In [ ]:
# Apply selected oversampling strategy to training data
if selected_sampling_strategy is not None:
    X_seq_tr, X_time_tr, X_num_tr, y_tr = oversample_sequences(
        X_seq_train,
        X_time_train,
        X_num_train,
        y_train,
        sampling_strategy=selected_sampling_strategy
    )
else:
    X_seq_tr = X_seq_train
    X_time_tr = X_time_train
    X_num_tr = X_num_train
    y_tr = y_train

print("===== Training Data =====")
print(f"Training samples: {len(y_tr)}")
print(f"Validation samples: {len(y_val)}")
print(f"Test samples: {len(y_test)}")
print(f"Training conversion rate: {np.mean(y_tr):.2%}")
print("\nReady for model training.")

In [ ]:
# Load selected hyperparameters for GRU with Touchpoint Attention
with open(
    os.path.join(save_path, "best_hp_ablation_2.json"),
    "r"
) as f:
    best_hp = json.load(f)

print("===== Selected Hyperparameters: GRU with Attention =====")

for key, value in best_hp.items():
    print(f"{key}: {value}")


In [ ]:
# Custom layers for touchpoint attention
@tf.keras.utils.register_keras_serializable()
class AttentionSoftmax(tf.keras.layers.Layer):
    """
    Apply softmax to attention scores across time steps.

    Input:  (batch, max_len, 1)
    Output: (batch, max_len, 1)
    """

    def call(self, inputs):
        scores = tf.squeeze(inputs, axis=-1)
        weights = tf.nn.softmax(scores, axis=-1)
        return tf.expand_dims(weights, axis=-1)


@tf.keras.utils.register_keras_serializable()
class WeightedSum(tf.keras.layers.Layer):
    """
    Sum weighted GRU outputs across time steps.

    Input:  (batch, max_len, units)
    Output: (batch, units)
    """

    def call(self, inputs):
        return tf.reduce_sum(inputs, axis=1)


print("Custom attention layers defined.")


In [ ]:

# Build GRU with Touchpoint Attention
tf.keras.backend.clear_session()

# Load selected hyperparameters
embedding_dim = best_hp["embedding_dim"]
gru_units = best_hp["gru_units"]
dropout = best_hp["dropout"]
learning_rate = best_hp["learning_rate"]
dense_units = best_hp["dense_units"]

# Model inputs
seq_input = Input(
    shape=(max_len,),
    name="seq_input"
)

time_input = Input(
    shape=(max_len, n_time_features),
    name="time_input"
)

num_input = Input(
    shape=(n_num_features,),
    name="num_input"
)

# Touchpoint embedding
embedding = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    mask_zero=True,
    name="embedding"
)(seq_input)

# Repeat journey-level numerical features across all time steps
num_repeated = tf.keras.layers.RepeatVector(
    max_len,
    name="num_repeated"
)(num_input)

# Combine sequential inputs
x = Concatenate(
    axis=-1,
    name="combined_input"
)([
    embedding,
    time_input,
    num_repeated
])

# GRU with sequence outputs
gru_out = GRU(
    units=gru_units,
    return_sequences=True,
    dropout=dropout,
    recurrent_dropout=dropout / 2,
    name="gru"
)(x)

# Touchpoint attention
energy_tp = Dense(
    units=gru_units // 2,
    activation="tanh",
    name="energy_tp"
)(gru_out)

score_tp = Dense(
    units=1,
    name="score_tp"
)(energy_tp)

a_tp = AttentionSoftmax(
    name="a_tp"
)(score_tp)

weighted_tp = Multiply(
    name="weighted_tp"
)([gru_out, a_tp])

v_tp = WeightedSum(
    name="v_tp"
)(weighted_tp)

# Dense representation
final_hidden = Dense(
    units=dense_units,
    activation="relu",
    name="final_hidden"
)(v_tp)

# Conversion probability
conv_output = Dense(
    units=1,
    activation="sigmoid",
    name="conversion"
)(final_hidden)

# Build model
model2 = Model(
    inputs=[
        seq_input,
        time_input,
        num_input
    ],
    outputs=conv_output,
    name="UniGRU_Modell2"
)

# Compile model
model2.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=learning_rate
    ),
    loss="binary_crossentropy",
    metrics=[
        AUC(name="roc_auc"),
        AUC(name="pr_auc", curve="PR")
    ]
)

# Display model architecture
model2.summary()

print("\nGRU with Touchpoint Attention initialized.")

In [ ]:
# Train GRU with Touchpoint Attention
checkpoint_path = os.path.join(
    save_path,
    "ablation_2_final.keras"
)

history2 = model2.fit(
    [X_seq_tr, X_time_tr, X_num_tr],
    y_tr,
    validation_data=(
        [X_seq_val, X_time_val, X_num_val],
        y_val
    ),
    epochs=50,
    batch_size=64,
    class_weight=best_cw_dict,
    callbacks=[
        EarlyStopping(
            monitor="val_pr_auc",
            patience=7,
            mode="max",
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_pr_auc",
            factor=0.5,
            patience=3,
            mode="max",
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_pr_auc",
            save_best_only=True,
            mode="max",
            verbose=1
        )
    ],
    verbose=1
)

print("\nGRU with Touchpoint Attention training completed.")
print(f"Epochs trained: {len(history2.history['loss'])}")


In [ ]:
# Save training history
history_path = os.path.join(
    save_path,
    "history2.pkl"
)

with open(history_path, "wb") as f:
    pickle.dump(history2.history, f)

print("Training history saved.")
print(f"Epochs trained: {len(history2.history['loss'])}")
print(f"Available metrics: {list(history2.history.keys())}")


In [ ]:
# Evaluation on the test set
y_pred_prob = model2.predict(
    [X_seq_test, X_time_test, X_num_test],
    verbose=0
).flatten()

pr_auc = average_precision_score(y_test, y_pred_prob)
roc_auc = roc_auc_score(y_test, y_pred_prob)

print("\n===== GRU with Touchpoint Attention: Test Evaluation =====")
print(f"PR-AUC:  {pr_auc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")


In [ ]:
# Save GRU with Touchpoint Attention evaluation results
results_abl2 = {
    "model": "UniGRU_Modell2",
    "pr_auc": float(pr_auc),
    "roc_auc": float(roc_auc)
}

results_path = os.path.join(
    save_path,
    "results_ablation_2.json"
)

with open(results_path, "w") as f:
    json.dump(results_abl2, f, indent=2)

# Save predicted conversion probabilities
predictions_path = os.path.join(
    save_path,
    "y_prob_ablation_2.npy"
)

np.save(predictions_path, y_pred_prob)

print("Evaluation results saved.")
print("Predicted probabilities saved.")
